**예제 1**

In [7]:
class MyStack:
    def pop(self):
        return "Removed"

def remove_last(obj):
    return obj.pop()

print(remove_last([1, 2, 3]))
print(remove_last(MyStack()))

3
Removed


**예제 2**

In [2]:
class MySequence:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

seq = MySequence([10, 20, 30])

print(len(seq))
print(seq[1])

for item in seq:
    print(item)

3
20
10
20
30


**예제 3**

In [3]:
class Foo:
    def __getitem__(self, pos):
        return range(0, 30, 10)[pos]

f = Foo()
print("f[1]:", f[1])
print("for loop:")
for i in f:
    print(i)

print("20 in f:", 20 in f)
print("15 in f:", 15 in f)


f[1]: 10
for loop:
0
10
20
20 in f: True
15 in f: False


**예제 4**

In [6]:
"""
PyObject *
PyObject_GetIter(PyObject *o)
{
    PyTypeObject *t = Py_TYPE(o);
    getitemfunc f;

    f = t->tp_iter;
    if (f == NULL) {
        if (PySequence_Check(o))
            return PySeqIter_New(o);
        return type_error("'%.200s' object is not iterable", o);
    }
    else {
        PyObject *res = (*f)(o);
        if (res != NULL && !PyIter_Check(res)) {
            PyErr_Format(PyExc_TypeError,
                "'%.iter__() must return an iterator, not %T'",
                o, res);
            Py_SETREF(res, NULL);
        }
        return res;
    }
}
"""

'\nPyObject *\nPyObject_GetIter(PyObject *o)\n{\n    PyTypeObject *t = Py_TYPE(o);\n    getitemfunc f;\n\n    f = t->tp_iter;\n    if (f == NULL) {\n        if (PySequence_Check(o))\n            return PySeqIter_New(o);\n        return type_error("\'%.200s\' object is not iterable", o);\n    }\n    else {\n        PyObject *res = (*f)(o);\n        if (res != NULL && !PyIter_Check(res)) {\n            PyErr_Format(PyExc_TypeError,\n                "\'%.iter__() must return an iterator, not %T\'",\n                o, res);\n            Py_SETREF(res, NULL);\n        }\n        return res;\n    }\n}\n'

**예제 5**

In [19]:
import collections

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

**예제 6**

In [14]:
import random

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

deck = FrenchDeck()

**예제 7**

In [18]:
def set_card(deck, position, card):
    deck._cards[position] = card

FrenchDeck.__setitem__ = set_card
random.shuffle(deck)
print(deck[:5])

[Card(rank='6', suit='clubs'), Card(rank='K', suit='clubs'), Card(rank='3', suit='hearts'), Card(rank='10', suit='hearts'), Card(rank='J', suit='hearts')]


**예제 8**

In [17]:
import collections
Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck2(collections.abc.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                                        for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    def __setitem__(self, position, value):
        self._cards[position] = value

    def __delitem__(self, position):
        del self._cards[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

**예제 9**

In [15]:
from abc import ABC, abstractmethod

class Drawable(ABC):
    @abstractmethod
    def draw(self):
        pass

class ExternalPainter:
    def draw(self):
        print("Drawing")

Drawable.register(ExternalPainter)
obj = ExternalPainter()
print("isinstance(obj, Drawable):", isinstance(obj, Drawable))

isinstance(obj, Drawable): True


**에제 10**

In [16]:
from collections.abc import Sized

class MyCollection:
    def __len__(self):
        return 10

Sized.register(MyCollection)

__main__.MyCollection

**예제 11**

In [20]:
import abc

class Tombola(ABC):
    @abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abstractmethod
    def pick(self):
        """Remove item at random, returning it.
        This method should raise `LookupError` when the instance is empty.
        """

    def loaded(self):
        """Return `True` if there's at least 1 item, `False` otherwise."""
        return bool(self.inspect())

    def inspect(self):
        """Return a sorted tuple with the items currently inside."""
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))

**에제 12**

In [23]:
class Tombola(ABC):
    @abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abstractmethod
    def pick(self):
        """Remove item at random, returning it."""

    def loaded(self):
        """Return True if there's at least 1 item, False otherwise."""
        return bool(self.inspect())

    def inspect(self):
        """Return a sorted tuple with the items currently inside."""
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(sorted(items))


In [25]:
import random

class BingoCage(Tombola):
    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):
        return self.pick()

**예제 13**

In [26]:
import random

class LotteryBlower(Tombola):
    def __init__(self, iterable):
        self._balls = list(iterable)

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty LotteryBlower')
        return self._balls.pop(position)

    def loaded(self):
        return bool(self._balls)

    def inspect(self):
        return tuple(sorted(self._balls))


**예제 14**

In [28]:
@Tombola.register
class TomboList(list):
    def pick(self):
        if self:
            position = random.randrange(len(self))
            return self.pop(position)
        else:
            raise LookupError('pop from empty TomboList')

    load = list.extend

    def loaded(self):
        return bool(self)

    def inspect(self):
        return tuple(sorted(self))

**에제 15**

In [30]:
TEST_FILE = 'tombola_tests.rst'
TEST_MSG = '{0:16} {1.attempted:2} tests, {1.failed:2} failed - {2}'

def main(argv):
    verbose = '-v' in argv
    real_subclasses = Tombola.__subclasses__()
    virtual_subclasses = list(getattr(Tombola, '_abc_registry', getattr(Tombola, '_abc_impl', set())))

    for cls in real_subclasses + list(virtual_subclasses):
        test(cls, verbose)

def test(cls, verbose=False):
    try:
        res = doctest.testfile(
            TEST_FILE,
            globs={'ConcreteTombola': cls},
            verbose=verbose,
            optionflags=doctest.REPORT_ONLY_FIRST_FAILURE)
        tag = 'FAIL' if res.failed else 'OK'
        print(TEST_MSG.format(cls.__name__, res, tag))
    except FileNotFoundError:
        print(f"[{cls.__name__}] 로드 완료 (실행 테스트를 위한 '{TEST_FILE}' 파일이 필요합니다)")

if __name__ == '__main__':
    bingo = BingoCage([1, 2, 3, 4, 5])
    print("BingoCage pick:", bingo())

    lotto = LotteryBlower([10, 20, 30])
    print("LotteryBlower inspect:", lotto.inspect())

    tombolist = TomboList([100, 200, 300])
    print("TomboList is subclass of Tombola:", issubclass(TomboList, Tombola))

BingoCage pick: 4
LotteryBlower inspect: (10, 20, 30)
TomboList is subclass of Tombola: True
